# Local volatility

How the local volatility model works, step by step, on TICKER1:

1. **the problem**: one Black–Scholes vol per option is not a model;
2. **the idea** (Dupire): the option prices fix the distribution of the stock at every maturity, and there is exactly
   one diffusion with a volatility $\sigma(t, S)$ that has those distributions;
3. **the calibration, by hand**: from the implied vol nodes to the local vol surface, every intermediate quantity
   computed and plotted, then compared with `LocalVolatilityModel`;
4. **what the model implies**: sanity checks, implied vol as an average of local vol;
5. **Monte Carlo**: simulation, forward, smile repriced, forward smile, greeks by AAD.

Docs: *Models and Pricing* (Models: Local volatility), *Architecture* (Models).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq
from scipy.optimize import brentq
from scipy.stats import norm

## The calibration in a nutshell

**Input**: the implied vol surface, a table of quotes $\sigma_{ij}$ (here 8 maturities $\times$ 9 strikes).
**Output**: a function $\sigma_{\mathrm{loc}}(t, S)$, the vol the stock has when it is at level $S$ at time $t$, stored on a grid.

There is **no fitting and no optimisation**: the local vol is *computed* from the quotes, and by construction the model
reprices every quote. The idea in one sentence: the implied vols tell how much variance the market expects to
accumulate up to each maturity; the local vol is the **rate** at which that variance accumulates, at each level of
the spot, corrected for the shape of the smile.

The four operations:

1. **Rewrite every quote** as a point of a curve: moneyness $y = \ln(K/F(T))$ (strike relative to the forward) and total
   variance $w = \sigma^2 T$ (the variance accumulated up to $T$).
2. **Join the quotes of each maturity with a smooth curve** $w(y)$ (a spline): now $w$, its slope (the skew) and its
   curvature are known at any moneyness, not only at the 9 strikes.
3. **Join the maturities**: at fixed moneyness, $w$ is linear between two maturities, so the variance added per unit of
   time, $\partial_T w$, is constant between them.
4. **Apply the formula at every point** $(t, y)$ of a grid:
   $$\sigma_{\mathrm{loc}}^2 = \frac{\text{variance added per unit of time } (\partial_T w)}{\text{shape factor } g \text{ (from the slope and curvature of the smile)}}$$
   and check that both are positive (otherwise the quotes have an arbitrage and no local vol exists).

```
quotes (T_i, K_j, vol_ij) ──1──> points (y_ij, w_ij) ──2──> smooth smile w(y) per maturity
                                                       ──3──> w(y, T) between maturities
                                                       ──4──> local vol(t, y) = sqrt( dw/dT / g )
```

Where this formula comes from (Dupire) is section 2; the four operations, one by one with plots, are section 3; then
what the model does with it (sections 4-5).

## 1. The problem

The market quotes an implied (Black–Scholes) vol $\sigma_{\mathrm{imp}}(K, T)$ for every strike and maturity. Each number is the
vol that makes Black–Scholes reproduce **one** option price: a 90% put and a 110% call of the same maturity use two
different vols, i.e. two different models of the same stock. That is fine to quote vanillas, but an exotic (a barrier, an
autocallable) depends on many strikes and dates at once and needs **one** model that reprices all the vanillas together.

Local volatility is the simplest such model: the vol is a deterministic function of time and spot,
$$\frac{\mathrm{d}S_t}{S_t} = \mu_t\,\mathrm{d}t + \sigma_{\mathrm{loc}}(t, S_t)\,\mathrm{d}W_t .$$
When the stock falls, its vol rises (if $\sigma_{\mathrm{loc}}$ is higher on the left): that is how a negative skew is produced.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES = np.array(vs.maturity, dtype=float)
K_NODES = np.array(vs.strike, dtype=float)
IV = vs.volatility_matrix.numpy()                      # [maturities, strikes]


def date_of(T):
    return EVALUATION_DATE + timedelta(days=max(1, int(round(T * 365))))


F_NODES = np.array([float(eq.forward(date_of(T))) for T in T_NODES])
print(f"{TICKER} spot {S0:.2f}, dividends: {eq.dividend_model}")
print("implied vols (%), maturity x strike / spot:")
print(pd.DataFrame(100 * IV, index=np.round(T_NODES, 2), columns=[f"{k:.0%}" for k in K_NODES / S0]).round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
for i in (0, 1, 3, 5, 7):
    ax.plot(K_NODES / F_NODES[i], 100 * IV[i], "o-", ms=4, label=f"T = {T_NODES[i]:.2f}")
ax.set(xlabel="K / F(T)", ylabel="implied vol (%)", title="The smile: one Black-Scholes vol per option")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## 2. The idea: option prices fix the distribution at every maturity

### Notation
| symbol | meaning |
|---|---|
| $F(T)$ | market forward to $T$ (rates, repo, dividends) |
| $X_T = S_T / F(T)$ | forward ratio: a martingale with $X_0 = 1$ |
| $k = K / F(T)$ | strike in moneyness |
| $c(k, T) = \mathbb{E}\big[(X_T - k)^+\big] = C(K,T) / \big(P(0,T)\,F(T)\big)$ | call, undiscounted, per unit of forward |
| $p_T(x)$ | risk-neutral density of $X_T$ |

Why $X$ and not $S$: under the risk-neutral measure the forward is the expected future spot, so $\mathbb{E}[X_T] = 1$ whatever
the rates and dividends. All the deterministic part (rates, repo, dividends) stays in $F(T)$; the model only has to
describe the uncertainty around it, $\mathrm{d}X_t / X_t = \sigma_{\mathrm{loc}}(t, X_t)\,\mathrm{d}W_t$.

### Step A: from prices to distributions (Breeden–Litzenberger, 1978)
A call is the average of its payoff over the final distribution:
$$c(k, T) = \int_k^\infty (x - k)\,p_T(x)\,\mathrm{d}x .$$
Moving the strike up by a little, I lose one unit on every path that ends above $k$:
$$\partial_k c = -\,\mathbb{P}(X_T > k),$$
and differentiating once more,
$$\partial_{kk} c = p_T(k).$$
**Intuition.** A very tight butterfly (long the calls at $k - h$ and $k + h$, short two at $k$, divided by $h^2$) pays a little
triangle of area one around $k$: it pays about 1 only if $X_T$ ends near $k$. Its price is therefore the probability of
ending there:
$$\frac{c(k - h) - 2\,c(k) + c(k + h)}{h^2} \;\longrightarrow\; p_T(k).$$
So **the calls of one maturity are a distribution of the stock at that maturity**, and every quoted maturity gives one.

### Step B: from the distributions to the local vol (Dupire, 1994)

**B.1 What is still missing.** Step A gives one distribution of $X_T$ per maturity: a photograph of where the stock can
be at each $T$. It says nothing about how the stock moves *between* two maturities, and to simulate a path a dynamics
is needed. Many different processes have the same distributions at all maturities (a stochastic vol model and the
local vol model below can reprice exactly the same calls and still move very differently in between), so the calls
alone do not pick a process.

**B.2 Dupire's choice.** Restrict the search to the simplest kind of process, with a vol that depends only on time and on
the level of the stock:
$$\frac{\mathrm{d}X_t}{X_t} = \sigma(t, X_t)\,\mathrm{d}W_t .$$
Dupire shows that in this class there is **one and only one** $\sigma(t, x)$ that reprices all the calls, and gives it in
closed form from the call prices. This is why the "calibration" of local vol is a computation, not an optimisation.
(Working with $X = S/F$ removes rates, repo and dividends: on $S$ the same formula has extra drift terms.)

**B.3 The derivation, step by step.** The idea: look at how much a call gains if its maturity is extended by $\mathrm{d}T$.

1. The call is the average of its payoff at $T$:
   $$c(k, T) = \mathbb{E}\big[f(X_T)\big], \qquad f(x) = (x - k)^+ .$$
2. The derivatives of the payoff in $x$:
   $$f'(x) = \mathbf{1}\{x > k\} \quad (0 \text{ below the strike, } 1 \text{ above}), \qquad f''(x) = \delta(x - k).$$
   The payoff is a straight line on each side of $k$: all its convexity sits in the corner at $k$, so the second
   derivative is a Dirac spike there. (It is the tight butterfly of Step A seen from the payoff side: the second
   difference of $(x - k)^+$ in $k$, divided by $h^2$, is a triangle of area one that shrinks to $\delta(x - k)$.)
3. Itô's formula for $f(X_t)$ between $T$ and $T + \mathrm{d}T$ (in the form that allows the corner: Tanaka):
   $$\mathrm{d}f(X_T) = f'(X_T)\,\mathrm{d}X_T + \tfrac12 f''(X_T)\,(\mathrm{d}X_T)^2,
   \qquad (\mathrm{d}X_T)^2 = \sigma^2(T, X_T)\,X_T^2\,\mathrm{d}T,$$
   that is
   $$\mathrm{d}(X_T - k)^+ = \underbrace{\mathbf{1}\{X_T > k\}\,\mathrm{d}X_T}_{\text{linear part}}
   \;+\; \underbrace{\tfrac12\,\delta(X_T - k)\,\sigma^2(T, X_T)\,X_T^2\,\mathrm{d}T}_{\text{convexity of the corner}} .$$
4. Take the expectation of each term:
   - linear part: $\mathbb{E}\big[\mathbf{1}\{X_T > k\}\,\mathrm{d}X_T\big] = 0$, because $X$ is a martingale ($\mathrm{d}X$ has zero mean whatever
     happened before). The straight parts of the payoff add no value over time.
   - corner part: the expectation of a Dirac times a function $h$ picks the value of $h$ at $k$, weighted by the
     probability of being there:
     $$\mathbb{E}\big[\delta(X_T - k)\,h(X_T)\big] = \int \delta(x - k)\,h(x)\,p_T(x)\,\mathrm{d}x = h(k)\,p_T(k),$$
     with $h(x) = \tfrac12\sigma^2(T, x)\,x^2$.
5. Hence the growth of the call with the maturity:
   $$\partial_T c(k, T) = \tfrac12\,\sigma^2(T, k)\,k^2\,p_T(k).$$
6. Step A gives the density from the calls, $p_T(k) = \partial_{kk} c$. Substitute and solve for $\sigma^2$:
   $$\boxed{\;\sigma_{\mathrm{loc}}^2(T, k) = \frac{\partial_T c(k, T)}{\tfrac12\,k^2\,\partial_{kk} c(k, T)}\;}$$
   On the right there are only market prices: the numerator is how the calls grow with the maturity (a calendar
   spread), the denominator is the density (the butterfly of Step A).

**B.4 Why only the paths at the strike count.** Over a short $\mathrm{d}T$ the stock moves by about $\pm\,\sigma k \sqrt{\mathrm{d}T}$ around
$k$. A path far below the strike stays worthless, a path far above it moves along the straight part of the payoff
and gains nothing on average. Only the paths within about $\sigma k\sqrt{\mathrm{d}T}$ of the strike can gain from the corner:
- the probability of being there is about $p_T(k) \times \sigma k\sqrt{\mathrm{d}T}$;
- what each of them gains on average is about $\sigma k\sqrt{\mathrm{d}T}$ (it ends above the corner half of the time);
- the product is of the order of $\sigma^2 k^2\, p_T(k)\,\mathrm{d}T$: the formula of step 5 (Itô gives the exact factor $\tfrac12$).

So the growth of the price with $T$ is
$$\tfrac12 \times (\text{local variance at the strike}) \times k^2 \times (\text{density at the strike}),$$
and knowing the calls I know the growth (left side) and the density: the local variance at $(T, k)$ is **determined**.

**B.5 It is the theta–gamma relation of Black–Scholes, in strike.** In Black–Scholes a hedged option earns on gamma what
it loses in theta. Dupire is the same relation with the strike in place of the spot:

| | Black–Scholes (in spot, backward) | Dupire (in strike, forward) |
|---|---|---|
| relation | $\Theta = -\tfrac12\,\sigma^2 S^2\,\Gamma$ | $\partial_T c = \tfrac12\,\sigma^2 k^2\,\partial_{kk} c$ |
| curvature | gamma $\partial_{SS} V$ | density $\partial_{kk} c$ |
| what varies | the spot, for one option | strike and maturity, for one spot (today) |
| known | $\sigma$, unknown price | prices, unknown $\sigma$ |

Black–Scholes uses the vol to get the price; Dupire reads the equation backwards and uses the prices of all strikes
and maturities to get the vol.

**B.6 A number.** At $k = 1$, if the calls grow by $\partial_T c = 0.02$ per year and the density there is $p_T(1) = 1.6$:
$$\sigma_{\mathrm{loc}}^2 = \frac{0.02}{\tfrac12 \times 1^2 \times 1.6} = 0.025, \qquad \sigma_{\mathrm{loc}} = 15.8\%.$$
A higher density at the strike with the same growth means a lower local vol: the growth is spread over more paths.

**B.7 What the formula needs, and why it is fragile.** It needs, at every $(T, k)$, a first derivative in maturity and a
**second derivative in strike** of call prices that the market quotes only at a few nodes. Second derivatives amplify
every kink of the interpolation between the nodes, and where the density is small (wings, short maturities) the ratio
divides by almost zero. The next cell applies the formula as it is, with finite differences on the interpolated
prices, to show exactly this; section 3 then fixes it by rewriting the formula in implied total variance with a smooth
spline.

### Step C: what the local variance means (Gyöngy, 1986; Derman–Kani, 1994)
If the true vol $\sigma_t$ is itself random (stochastic vol), the model that reprices the same calls has
$$\sigma_{\mathrm{loc}}^2(T, k) = \mathbb{E}\big[\sigma_T^2 \,\big|\, X_T = k\big]:$$
the local variance is the average of the instantaneous variance over the paths that go through $(T, k)$
(Gatheral, eq. 1.12). This is also the basis of the stochastic-local vol model, whose leverage is
$L^2 = \sigma_{\mathrm{loc}}^2 / \mathbb{E}[v \mid X]$.

**References**: Breeden and Litzenberger, *Prices of state-contingent claims implicit in option prices*, Journal of
Business (1978); Dupire, *Pricing with a smile*, Risk (1994); Derman and Kani, *Riding on a smile*, Risk (1994);
Gyöngy, *Mimicking the one-dimensional marginal distributions of processes having an Itô differential*, Probability
Theory and Related Fields (1986); Gatheral, *The Volatility Surface* (2006), ch. 1.

### Try it directly on the market prices
Below: the calls of the surface on a fine grid of $k$ at 1Y, the density by the butterfly of Step A, and the local vol by
the formula of Step B with finite differences.

In [ ]:
def iv_at(K, T):
    # implied vol of the market surface (bilinear in strike and maturity, as the closed-form pricers use)
    return float(vs.volatility(K, T))


def c_undisc(k, T):
    # E[(X_T - k)^+] from the surface: Black-Scholes per unit forward with the vol at K = k F(T)
    F = float(eq.forward(T))
    s = iv_at(k * F, T) * np.sqrt(T)
    d1 = -np.log(k) / s + 0.5 * s
    return norm.cdf(d1) - k * norm.cdf(d1 - s)


T1, dT, dk = 1.0, 0.02, 0.005
ks = np.arange(0.62, 1.38, dk)
c0 = np.array([c_undisc(k, T1) for k in ks])
c_up = np.array([c_undisc(k, T1 + dT) for k in ks])
c_dn = np.array([c_undisc(k, T1 - dT) for k in ks])
density = (c0[2:] - 2 * c0[1:-1] + c0[:-2]) / dk ** 2           # Breeden-Litzenberger: d2c/dk2
dcdT = (c_up - c_dn)[1:-1] / (2 * dT)
dupire = np.sqrt(np.maximum(dcdT / (0.5 * ks[1:-1] ** 2 * density), 0))

atm = iv_at(float(eq.forward(T1)), T1)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(ks[1:-1], density, label="market density (d2c/dk2)")
axes[0].plot(ks[1:-1], norm.pdf(np.log(ks[1:-1]) / (atm * np.sqrt(T1)) + 0.5 * atm * np.sqrt(T1)) / (ks[1:-1] * atm * np.sqrt(T1)),
             "--", label=f"lognormal, ATM vol {atm:.1%}")
axes[0].set(xlabel="k = K / F", title="1Y risk-neutral density of X")
axes[0].legend(fontsize=8)
axes[1].plot(ks[1:-1], 100 * dupire, label="Dupire on prices (finite differences)")
axes[1].plot(ks[1:-1], [100 * iv_at(k * float(eq.forward(T1)), T1) for k in ks[1:-1]], label="implied vol")
axes[1].set(xlabel="k = K / F", ylabel="vol (%)", title="Local vol at 1Y, directly from the prices", ylim=(10, 60))
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

This does not work. The surface is interpolated **linearly in vol** between the strike nodes: the slope of the smile
jumps at every node, so the second derivative of the prices (the density) has a **spike** at each node strike and is too
low in between. The "density" is not a sensible distribution, and the local vol inherits the spikes. Finite
differences of prices make it worse in the wings and at short maturities, where the prices are tiny. The formula is
right; the representation of the surface is not smooth enough for it. The calibration below fixes this by working in
**total implied variance** with a **smooth spline**.

## 3. Calibration, step by step

### 3.0 What "calibrating" means here
Dupire's formula (Step B) says that the local vol at $(T, k)$ is **fixed** by the call prices around that point. So there
is no parameter to fit: if the call prices were known for *every* strike and *every* maturity, the local vol would
follow by applying the formula. The problem is that the market gives only a table of quotes (here 8 maturities
$\times$ 9 strikes = 72 implied vols), while the formula needs, at every $(T, k)$, a derivative in maturity and a second
derivative in strike.

So calibrating local vol means: **build, from the 72 quotes, a smooth surface defined everywhere, then apply the
formula to it.** The only real choice is *how* to fill the gaps between the quotes; section 2 showed that a careless
choice (linear in vol) gives a spiky density and a useless local vol. Everything below is about making that choice well.

### 3.1 Why change variables: $y = \ln(K/F(T))$ and $w = \sigma_{\mathrm{imp}}^2 T$
Instead of call prices $c(k, T)$ we work with the **total implied variance** $w(y, T) = \sigma_{\mathrm{imp}}^2(y, T)\,T$ as a function of the
**forward log-moneyness** $y = \ln(K/F(T)) = \ln k$. Four reasons:

1. **It is what the market quotes.** Implied vols are smooth and of similar size across strikes; call prices range from
   almost the forward to almost zero, and their second derivatives in the wings are tiny numbers, very sensitive to
   rounding.
2. **The Black–Scholes price depends only on $y$ and $w$**: per unit forward, $c = N(d_+) - e^{y} N(d_-)$ with
   $d_\pm = -y/\sqrt w \pm \sqrt w / 2$. Rates, dividends and the maturity itself have disappeared into $F(T)$ and $w$.
3. **$w$ adds up over time.** Without smile, $w(T) = \int_0^T \sigma^2(t)\,\mathrm{d}t$: the variance accumulated up to $T$. Its growth
   between two maturities is the variance of that period (the forward variance), which is what a vol per unit of time
   should be.
4. **$y$ follows the forward.** The model lives on $X = S/F$, so a fixed $y$ is a fixed point of the model for all $t$; a
   fixed strike $K$ is not (its moneyness moves with the forward). The derivative $\partial_T$ in Dupire is taken at fixed $k$,
   i.e. at fixed $y$.

### 3.2 From Dupire to Gatheral: the formula in $(y, w)$
Write the market call as the Black–Scholes price with the market total variance: $c(y, T) = c_{\mathrm{BS}}\big(y, w(y, T)\big)$.
In $y$ the Dupire formula reads ($k^2 \partial_{kk} = \partial_{yy} - \partial_y$):
$$\sigma_{\mathrm{loc}}^2 = \frac{\partial_T c}{\tfrac12\,(\partial_{yy} c - \partial_y c)} .$$
- **Numerator.** At fixed $y$, $T$ enters only through $w$, so by the chain rule
  $$\partial_T c = \frac{\partial c_{\mathrm{BS}}}{\partial w}\,\partial_T w .$$
- **Denominator.** Here $y$ enters both directly and through $w(y)$, so the chain rule produces terms in $\partial_y w$ and
  $\partial_{yy} w$. After using the Black–Scholes identities for the derivatives of $c_{\mathrm{BS}}$ (Gatheral, pp. 11–13), it comes
  out as the same factor times a correction:
  $$\tfrac12\,(\partial_{yy} c - \partial_y c) = \frac{\partial c_{\mathrm{BS}}}{\partial w}\; g,\qquad
  g = 1 - \frac{y}{w}\,\partial_y w + \frac14\Big(-\frac14 - \frac1w + \frac{y^2}{w^2}\Big)(\partial_y w)^2 + \frac12\,\partial_{yy} w .$$
- **Ratio.** The common factor $\partial c_{\mathrm{BS}} / \partial w$ (the vega per unit of total variance) cancels:
  $$\boxed{\;\sigma_{\mathrm{loc}}^2(T, y) = \frac{\partial_T w(y, T)}{g(y, T)}\;}\qquad\text{(Gatheral, eq. 1.10)}$$

Check without smile: $w$ does not depend on $y$, so $g = 1$ and $\sigma_{\mathrm{loc}}^2 = \partial_T w$. The denominator identity becomes
$\tfrac12(\partial_{yy} - \partial_y)\,c_{\mathrm{BS}} = \partial_w c_{\mathrm{BS}}$: the Black–Scholes equation in total variance, i.e. the theta–gamma relation
of B.5. A flat surface gives a flat local vol, and a vol constant in strike but rising with maturity gives the forward
vol between maturities.

### 3.3 The two pieces
**Numerator $\partial_T w$: how much variance the market adds per unit of time at that moneyness.** Example: at the money,
6M implied vol 20% and 1Y 22% give $w = 0.20^2 \times 0.5 = 0.0200$ and $w = 0.22^2 \times 1 = 0.0484$, so between 6M and 1Y
$\partial_T w = (0.0484 - 0.0200)/0.5 = 0.0568$: a forward vol of $\sqrt{0.0568} = 23.8\%$, higher than both implied vols,
because the 1Y implied vol is the average over the whole year and the first half was quieter.

**Denominator $g$: the smile correction, and it is the density.** With $d_- = -y/\sqrt w - \sqrt w/2$, the risk-neutral density
of $y = \ln X_T$ is
$$p(y) = g(y)\;\times\;\underbrace{\frac{1}{\sqrt{2\pi w}}\,e^{-d_-^2/2}}_{\text{lognormal density with total variance } w(y)} .$$
So $g$ says how much more (or less) likely the market makes the level $y$ than a lognormal with the same variance. Where
the market puts more mass ($g > 1$), less local variance is needed to produce the same growth of the calls (B.6), and
vice versa. Its terms:
- $-\frac{y}{w}\,\partial_y w$: the **skew**. With a negative skew ($\partial_y w < 0$) it is $< 1$ on the downside ($y < 0$): lower $g$, higher
  local vol. This is the main reason why the local vol on the downside is above the implied vol.
- $\frac14(\dots)(\partial_y w)^2$: second-order skew effect, small near the money.
- $\frac12\,\partial_{yy} w$: the **curvature** of the smile; a convex smile adds mass in the region.

**Two conditions**, which are both the no-static-arbitrage conditions and the conditions for a local vol to exist:
- $\partial_T w \ge 0$ at fixed $y$: otherwise a calendar spread (long the longer maturity, short the shorter) costs less than zero;
- $g > 0$: otherwise the density is negative, i.e. a butterfly has a negative price (Step A).

### 3.4 What the surface $w(y, T)$ must satisfy
The formula needs $w$, $\partial_y w$, $\partial_{yy} w$ and $\partial_T w$ at every point, from 72 numbers. The construction has to:

| requirement | why |
|---|---|
| pass through every quote | the model must reprice the quotes |
| continuous $\partial_y w$ and $\partial_{yy} w$ in $y$ | $g$ contains them: a jump in the slope (linear interpolation) gives a spike in the density and in the local vol (section 2) |
| derivatives in closed form | finite differences of interpolated data amplify noise |
| $\partial_T w \ge 0$ whenever the quotes allow it | otherwise $\sigma_{\mathrm{loc}}^2 < 0$ |
| sensible wings beyond the last strikes | the simulated paths go there; $w$ cannot grow faster than $2\lvert y \rvert$ (Lee, 2004) |
| differentiable in the 72 vols | the greeks (vega per node) flow through the calibration by AAD |

### 3.5 The choices
| step | what | why this choice |
|---|---|---|
| 1 | each quote to $(y_{ij}, w_{ij})$ | the coordinates of the formula (3.1) |
| 2 | per maturity, natural cubic spline of $w$ in $y$; linear wings, slope $\le 1.9$ | passes through the quotes, continuous $\partial_y w$ and $\partial_{yy} w$ in closed form; "natural" = the smoothest curve through the points (zero curvature at the ends); wings below Lee's bound 2 |
| 3 | at fixed $y$, $w$ linear in $T$ between maturities | the simplest choice that keeps $\partial_T w \ge 0$: if the slices do not cross, every segment goes up; $\partial_T w$ is then the forward variance of the period |
| 4 | the formula on a grid in $(t, y)$, $g \ge 10^{-3}$, $\sigma_{\mathrm{loc}} \le 200\%$ | the Monte Carlo needs $\sigma_{\mathrm{loc}}(t, S)$ millions of times: computed once on a grid, then read by interpolation ($y = \ln(S/F(t))$); floors only protect against bad data |
| 5 | check $g > 0$, $\partial_T w > 0$ where there are quotes | if it fails the quotes have an arbitrage and no local vol reprices them |
| 6 | compare with `LocalVolatilityModel` | the library runs the same steps in TensorFlow |

A consequence of step 3 to keep in mind: $\partial_T w$ is constant between two maturities and jumps at each maturity, so the
local vol is piecewise constant in time (in its time dependence through $\partial_T w$) and changes at the quoted maturities.

### The whole calibration at one point
Before the steps on the whole grid, the four operations by hand at a single point: time $t = 0.75$ (between the 6M and
1Y maturities) and moneyness $y = -0.2$ (about 18% below the forward). What each line does:
1. the quotes of the two maturities around $t$ become points $(y, w)$;
2. a spline through each of the two slices gives $w$, $\partial_y w$, $\partial_{yy} w$ at $y = -0.2$;
3. between the two maturities everything is linear in time: weights $1-u$, $u$ with $u = (0.75 - 0.5)/(1 - 0.5)$, and
   $\partial_T w$ = (difference of $w$) / (difference of maturity);
4. $g$ from the formula of 3.2, and $\sigma_{\mathrm{loc}} = \sqrt{\partial_T w / g}$.

In [ ]:
t0, y0 = 0.75, -0.2
i_lo, i_hi = list(T_NODES).index(T_NODES[2]), list(T_NODES).index(1.0)        # the 6M and 1Y maturities
T_lo, T_hi = T_NODES[i_lo], T_NODES[i_hi]

# 1. quotes -> (y, w) for the two maturities
pts = {i: (np.log(K_NODES / F_NODES[i]), IV[i] ** 2 * T_NODES[i]) for i in (i_lo, i_hi)}
# 2. smooth smile per maturity: w, dw/dy, d2w/dy2 at y0
smile = {i: [float(v[0]) for v in tq.natural_cubic_spline(*pts[i], np.array([y0]), 1.9)] for i in (i_lo, i_hi)}
# 3. between the maturities: linear in T
u = (t0 - T_lo) / (T_hi - T_lo)
w, w_y, w_yy = [(1 - u) * a + u * b for a, b in zip(smile[i_lo], smile[i_hi])]
w_T = (smile[i_hi][0] - smile[i_lo][0]) / (T_hi - T_lo)
# 4. the formula
g = 1 - y0 / w * w_y + 0.25 * (-0.25 - 1 / w + (y0 / w) ** 2) * w_y ** 2 + 0.5 * w_yy
sigma_loc = np.sqrt(w_T / g)

print(f"w at 6M / 1Y        : {smile[i_lo][0]:.5f} / {smile[i_hi][0]:.5f}   (implied vols {np.sqrt(smile[i_lo][0] / T_lo):.2%} / {np.sqrt(smile[i_hi][0] / T_hi):.2%})")
print(f"w, dw/dy, d2w/dy2   : {w:.5f}, {w_y:.5f}, {w_yy:.5f}   (interpolated at t = {t0})")
print(f"variance added dw/dT: {w_T:.5f}   -> a 'forward vol' of {np.sqrt(w_T):.2%} without smile correction")
print(f"shape factor g      : {g:.4f}")
print(f"local vol           : sqrt({w_T:.5f} / {g:.4f}) = {sigma_loc:.2%}")
model = tq.LocalVolatilityModel.from_equity(eq)
S_point = float(eq.forward(t0)) * np.exp(y0)
print(f"LocalVolatilityModel at the same point: {float(model.sigma_loc(t0, [S_point])[0]):.2%}  (bilinear on its grid)")

### Step 1: the nodes in the formula's coordinates
Each node $(T_i, K_j, \sigma_{ij})$ becomes $y_{ij} = \ln(K_j / F(T_i))$ and $w_{ij} = \sigma_{ij}^2 T_i$.

**Why**: these are the variables of the formula (3.1). Note that the same strike has a different $y$ at every maturity,
because the forward moves with rates and dividends: the slices are not aligned in $y$, which is why step 2 builds a
curve per maturity, evaluated afterwards on a common $y$ grid.

In [ ]:
Y_NODES = np.log(K_NODES[None, :] / F_NODES[:, None])      # [nT, nK]
W_NODES = IV ** 2 * T_NODES[:, None]
i1 = list(T_NODES).index(1.0)
print(f"1Y slice: forward {F_NODES[i1]:.2f}")
print(pd.DataFrame({"K": K_NODES, "K/S0": K_NODES / S0, "y = ln(K/F)": Y_NODES[i1], "implied vol": IV[i1],
                    "w = vol^2 T": W_NODES[i1]}).round(4).to_string(index=False))

### Step 2: a smooth smile per maturity
Through the 9 points $(y_{ij}, w_{ij})$ of one maturity, a **natural cubic spline**: a cubic polynomial between two nodes,
glued so that the value, the slope and the curvature are continuous at every node, with zero curvature at the end
nodes. Among all curves through the points with continuous curvature, it is the one that bends least
($\int (\partial_{yy} w)^2\,\mathrm{d}y$ minimal).

**Why**: $g$ needs $\partial_y w$ and $\partial_{yy} w$, and with a spline they are exact polynomials, continuous everywhere: no spikes, no
finite differences. The curve passes through the quotes, so the quotes are repriced.

**Outside the nodes** (the far wings, which the simulated paths do reach) the spline is continued as a straight line,
with the slope capped at 1.9: Lee's moment formula says that $w$ cannot grow faster than $2\lvert y \rvert$ without arbitrage.
There $\partial_{yy} w = 0$.

Below, the 1Y slice: $w$ with the nodes, then its slope (the skew) and its curvature.

In [ ]:
y = np.linspace(-0.8, 0.6, 281)
w1, w1_y, w1_yy = (v.numpy() for v in tq.natural_cubic_spline(Y_NODES[i1], W_NODES[i1], y, 1.9))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, v, name in zip(axes, (w1, w1_y, w1_yy), ("w", "dw/dy", "d2w/dy2")):
    ax.plot(y, v)
    ax.axvspan(Y_NODES[i1][0], Y_NODES[i1][-1], color="grey", alpha=0.1)
    ax.set(title=f"1Y: {name}", xlabel="y")
    ax.grid(alpha=0.3)
axes[0].plot(Y_NODES[i1], W_NODES[i1], "k.", ms=7)
plt.show()
print("shaded: the strike nodes; outside, w is linear (dw/dy constant, d2w/dy2 = 0)")

In [ ]:
W = np.stack([tq.natural_cubic_spline(Y_NODES[i], W_NODES[i], y, 1.9)[0].numpy() for i in range(len(T_NODES))])
fig, ax = plt.subplots(figsize=(8, 4))
for i, T in enumerate(T_NODES):
    ax.plot(y, W[i], label=f"T = {T:.2f}")
    ax.plot(Y_NODES[i], W_NODES[i], "k.", ms=4)
ax.set(xlabel="y = ln(K / F)", ylabel="w = iv² T", title="All the slices: they must not cross (calendar arbitrage)")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.show()
print("min over y of w(T_i+1) - w(T_i):", np.round((W[1:] - W[:-1]).min(axis=1), 5), "(all > 0: no calendar arbitrage)")

### Step 3: between the maturities
At fixed $y$, $w$ is **linear in $T$** between two maturities:
$$w(y, t) = (1 - u)\,w(y, T_i) + u\,w(y, T_{i+1}),\quad u = \frac{t - T_i}{T_{i+1} - T_i},\qquad
\partial_T w = \frac{w(y, T_{i+1}) - w(y, T_i)}{T_{i+1} - T_i}.$$
The same weights apply to $\partial_y w$ and $\partial_{yy} w$ (the slices are blended, not re-splined).

**Why**: $\partial_T w$ is then the variance added by the market between $T_i$ and $T_{i+1}$ at that moneyness, per unit of time
(the forward variance of 3.3), and it is positive as long as the slices do not cross (checked in the plot of step 2).
A smoother interpolation in time could overshoot and produce $\partial_T w < 0$. The price is a local vol that jumps at
each maturity.

**Outside**: before the first maturity $w = (t / T_1)\,w(y, T_1)$ (implied vol flat back to 0); after the last one the
implied vol is kept flat ($w$ grows proportionally to $t$).

In [ ]:
def time_weights(t):
    # w(t) = a @ w_nodes, dw/dt = b @ w_nodes (linear in T between maturities, flat implied vol outside)
    a, b = np.zeros(len(T_NODES)), np.zeros(len(T_NODES))
    if t <= T_NODES[0]:
        a[0], b[0] = t / T_NODES[0], 1 / T_NODES[0]
    elif t >= T_NODES[-1]:
        a[-1], b[-1] = t / T_NODES[-1], 1 / T_NODES[-1]
    else:
        i = np.searchsorted(T_NODES, t, side="right") - 1
        h = T_NODES[i + 1] - T_NODES[i]
        a[i], a[i + 1] = 1 - (t - T_NODES[i]) / h, (t - T_NODES[i]) / h
        b[i], b[i + 1] = -1 / h, 1 / h
    return a, b


ts = np.linspace(0.01, 5.5, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for y0 in (0.0, -0.3):
    w_y0 = np.array([np.interp(y0, y, W[i]) for i in range(len(T_NODES))])
    ab = [time_weights(t) for t in ts]
    axes[0].plot(ts, [a @ w_y0 for a, _ in ab], label=f"y = {y0}")
    axes[0].plot(T_NODES, w_y0, "k.", ms=5)
    axes[1].step(ts, [b @ w_y0 for _, b in ab], where="post", label=f"y = {y0}")
axes[0].set(xlabel="T", title="w(y, T) at fixed y: piecewise linear")
axes[1].set(xlabel="T", title="dw/dT: constant between maturities")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

### Step 4: the local vol on a grid
Now $w$, $\partial_y w$, $\partial_{yy} w$ and $\partial_T w$ are known at any $(t, y)$, and the formula gives the local vol there.

**Why a grid**: the Monte Carlo asks for $\sigma_{\mathrm{loc}}(t, S)$ at every time step and on every path. The model computes the
formula once on a grid (300 points of $y \in [\ln 0.05, \ln 5]$, time steps of 1/96 year plus the maturities) and during
the simulation reads it by bilinear interpolation at $y = \ln(S_t / F(t))$. $g$ is floored at $10^{-3}$ and $\sigma_{\mathrm{loc}}$ capped at
200% only to survive bad data (step 5 says whether they bite).

Below, just before 1Y: the numerator, the denominator and their ratio.

In [ ]:
model = tq.LocalVolatilityModel.from_equity(eq)
_, t_lv, y_lv = (v.numpy() for v in model.local_vol_grid)       # the model's grid (we only reuse its points)

# step 2 on the grid y: spline values and derivatives per maturity
spl = [tq.natural_cubic_spline(Y_NODES[i], W_NODES[i], y_lv, 1.9) for i in range(len(T_NODES))]
Wn, Wn_y, Wn_yy = (np.stack([s[j].numpy() for s in spl]) for j in range(3))     # [nT, nY]
# step 3 in time
A = np.stack([time_weights(t)[0] for t in t_lv])
B = np.stack([time_weights(t)[1] for t in t_lv])
w, w_y, w_yy, w_t = np.maximum(A @ Wn, 1e-12), A @ Wn_y, A @ Wn_yy, B @ Wn
# step 4: the formula
g = 1 - y_lv / w * w_y + 0.25 * (-0.25 - 1 / w + (y_lv / w) ** 2) * w_y ** 2 + 0.5 * w_yy
sigma_hand = np.sqrt(np.clip(np.maximum(w_t, 1e-12) / np.maximum(g, 1e-3), 1e-12, 4.0))

k1 = int(np.argmin(np.abs(t_lv - 0.99)))                         # a time just before 1Y
sel = (y_lv > -0.8) & (y_lv < 0.6)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(y_lv[sel], w_t[k1, sel]); axes[0].set_title(f"numerator dw/dT at t = {t_lv[k1]:.2f}")
axes[1].plot(y_lv[sel], g[k1, sel]); axes[1].set_title("denominator g")
axes[2].plot(y_lv[sel], 100 * sigma_hand[k1, sel], label="local vol")
axes[2].plot(y_lv[sel], 100 * np.sqrt(np.interp(y_lv[sel], y, W[i1]) / T_NODES[i1]), label="1Y implied vol")
axes[2].set_title("sqrt(numerator / g)"); axes[2].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("y"); ax.grid(alpha=0.3)
plt.show()

**Why the two curves are on the same plot, and how to read it.** They are *not* the same object, so the plot is not a
fit to judge by eye:
- the **implied vol** $\sigma_{\mathrm{imp}}(K, T)$ is one number per option: the constant vol that, put in Black–Scholes from today to $T$,
  gives the price of *that* option;
- the **local vol** $\sigma_{\mathrm{loc}}(t, S)$ is the instantaneous vol of the stock when it is at $S$ at time $t$.

They are on the same axes because the first is (roughly) an **average of the second**: the implied vol at strike $K$ is
the local vol averaged along the paths that go from today's spot to $K$ at $T$ (made precise in section 4). An average
is flatter than what it averages, so the local vol must be steeper than the implied smile to produce it. The fit is
checked elsewhere: simulating with the local vol and inverting the option prices with Black gives back the market
implied vols (section 5, "The smile repriced").

The small kinks near the ends of the strike range are where the spline meets the linear wings ($\partial_{yy} w$ drops to 0
there).

### The rule of 2
For short maturities the average is close to a plain average of the local vol over the interval between the spot and
the strike. With a linear local skew $\sigma_{\mathrm{loc}}(y) \approx a + b\,y$ around the money:
$$\sigma_{\mathrm{imp}}(y) \approx \frac1y \int_0^y (a + b\,z)\,\mathrm{d}z = a + \frac b2\,y
\qquad\Longrightarrow\qquad
\frac{\partial \sigma_{\mathrm{imp}}}{\partial y}\Big|_{\mathrm{ATM}} \approx \frac12\,\frac{\partial \sigma_{\mathrm{loc}}}{\partial y}\Big|_{\mathrm{ATM}} :$$
**the local skew is about twice the implied skew** (Derman, Kani and Zou, 1996; Gatheral, ch. 3). For longer maturities
the average runs also over time, through the whole term structure, and the ratio moves away from 2. Below, the ATM
slopes $\partial\sigma/\partial y$ of the market and of the calibrated model (by central differences at $y = \pm 0.02$).

In [ ]:
h_y = 0.02
rows = []
for T in (0.25, 1.0, 3.0):
    F_T = float(eq.forward(date_of(T)))
    iv_y = lambda yy: float(vs.volatility(F_T * np.exp(yy), T))
    row = {"T": T, "implied ATM": iv_y(0.0), "implied skew": (iv_y(h_y) - iv_y(-h_y)) / (2 * h_y)}
    for label, t in (("T/2", T / 2), ("T", 0.98 * T)):
        F_t = float(eq.forward(date_of(t)))
        lv_y = lambda yy: float(model.sigma_loc(t, [F_t * np.exp(yy)])[0])
        row[f"local ATM at {label}"] = lv_y(0.0)
        row[f"local skew at {label}"] = (lv_y(h_y) - lv_y(-h_y)) / (2 * h_y)
    rows.append(row)
skews = pd.DataFrame(rows).set_index("T")
skews["ratio at T/2"] = skews["local skew at T/2"] / skews["implied skew"]
skews["ratio at T"] = skews["local skew at T"] / skews["implied skew"]
print(skews.round(3).to_string())

The ratio is close to 2 at short maturities and moves with the maturity, as expected. Note also the **level**: the local
vol at the money is below the implied ATM vol, for two reasons. The local vol is convex in $y$ (higher on both sides),
so its average along the paths is above its value at the money; and here the ATM term structure is decreasing, so the
forward variances (the numerator $\partial_T w$) are below the implied ones.

$g$ is the density term: with $d_- = -y/\sqrt w - \sqrt w/2$, the risk-neutral density of $y = \ln X_T$ is
$p(y) = \frac{g(y)}{\sqrt{2\pi w}}\,e^{-d_-^2/2}$. On the smooth representation it is a proper density, with the fat left tail of
the skew (compare with the spiky one of section 2).

In [ ]:
d_minus = -y / np.sqrt(w1) - 0.5 * np.sqrt(w1)
g1 = 1 - y / w1 * w1_y + 0.25 * (-0.25 - 1 / w1 + (y / w1) ** 2) * w1_y ** 2 + 0.5 * w1_yy
p_y = g1 / np.sqrt(2 * np.pi * w1) * np.exp(-0.5 * d_minus ** 2)
k_grid = np.exp(y)
lognormal = norm.pdf(y / (atm * np.sqrt(T1)) + 0.5 * atm * np.sqrt(T1)) / (atm * np.sqrt(T1))
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(k_grid, p_y / k_grid, label="from the spline: g / sqrt(2 pi w) exp(-d-^2 / 2)")
ax.plot(k_grid, lognormal / k_grid, "--", label=f"lognormal, ATM vol {atm:.1%}")
ax.set(xlabel="k = K / F", title="1Y risk-neutral density of X, smooth", xlim=(0.45, 1.8))
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()
print(f"integral of the density: {np.trapezoid(p_y, y):.4f} (the rest is beyond the plotted range)")

### Step 5: no-arbitrage check
Where the market data are ($y$ inside the strike nodes of every maturity), $g > 0$ (a positive density: no butterfly
arbitrage) and $\partial_T w > 0$ (no calendar arbitrage) must hold; otherwise no local vol can reprice the surface and the
model floors the formula and warns.

In [ ]:
in_data = (y_lv >= Y_NODES[:, 0].max()) & (y_lv <= Y_NODES[:, -1].min())
print(f"min g = {g[:, in_data].min():.3f}, min dw/dT = {w_t[:, in_data].min():.4f}  -> arbitrage-free")
print("model's own report:", model.arbitrage)

### Step 6: the same as the library
`LocalVolatilityModel.from_equity` runs exactly these steps (in TensorFlow, so that the greeks can flow through them).

In [ ]:
sigma = model.local_vol_grid[0].numpy()
print(f"max |local vol by hand - LocalVolatilityModel| = {np.abs(sigma_hand - sigma).max():.2e}")

fig, ax = plt.subplots(figsize=(8, 4))
mesh = ax.pcolormesh(y_lv[sel], t_lv, sigma[:, sel], shading="auto", cmap="viridis", vmax=1.0)
ax.plot(Y_NODES.ravel(), np.repeat(T_NODES, len(K_NODES)), "w.", ms=3)
fig.colorbar(mesh, label="local vol")
ax.set(xlabel="y = ln(S / F(t))", ylabel="t", title="The calibrated local vol surface (dots: implied vol nodes)")
plt.show()

The local vol is highest at short maturities on the downside: there the implied smile is steepest, and (section 4) the
implied vol is an *average* of local vols, so to produce a steep average the local vol must be steeper still.

### How local vol is calibrated in practice
The whole difficulty is the one of section 2: Dupire needs **second derivatives in strike and a first derivative in time
of a surface known only at a few nodes**, and the result is only as good as the interpolation between them. Smooth is
not enough: a spline can oscillate between the nodes and produce $g < 0$ (a negative density), and a careless time
interpolation can produce $\partial_T w < 0$. The common approaches:

1. **A parametric smile without arbitrage, then Gatheral's formula.** SVI per maturity (Gatheral, 2004) or SSVI for the
   whole surface (Gatheral and Jacquier, *Arbitrage-free SVI volatility surfaces*, Quantitative Finance, 2014): $w(y, T)$ is
   an explicit function with closed-form derivatives and explicit conditions for no butterfly and no calendar arbitrage.
   The de facto standard in equity. (The synthetic market of this notebook is generated with SSVI.)
2. **Interpolation of the prices under no-arbitrage constraints** (convex in the strike, increasing in the maturity), then
   Dupire on the interpolated prices: e.g. Fengler, *Arbitrage-free smoothing of the implied volatility surface*,
   Quantitative Finance (2009); Kahalé, *An arbitrage-free interpolation of volatilities*, Risk (2004).
3. **Fitting the local vol directly through the Dupire PDE** (Andreasen and Huge, *Volatility interpolation*, Risk, 2011):
   a piecewise constant local vol such that one implicit finite-difference step of the forward PDE reprices the quotes
   of each maturity exactly. Arbitrage-free by construction, robust with sparse or noisy quotes, widely used on trading
   desks. Related: least squares with a smoothness penalty on the local vol (regularisation).
4. Pricing then usually runs on the **same PDE grid** (vanillas, barriers), Monte Carlo being kept for path-dependent
   products such as autocallables.

**What this library does**: approach 1 in spirit, but non-parametric. A natural cubic spline in total variance per
maturity (wings capped by Lee's bound), linear in time, the formula on a grid, and an explicit **check** of
$g > 0$ and $\partial_T w > 0$ (with a warning) instead of a guarantee. Note the inconsistency it leaves: the closed-form pricers
interpolate the same nodes linearly in vol, so between the nodes the two models see slightly different smiles (a
digital priced off the smile slope shows it most).

## 4. What the model implies
### Sanity checks
1. **Skew**: the local vol smile is steeper than the implied one. The rule of thumb is a local skew **twice** the implied
   one at the money ($\partial_y \sigma_{\mathrm{imp}} \approx \frac12 \partial_y \sigma_{\mathrm{loc}}$: the implied vol averages the local vol between the spot
   and the strike); it holds for a linear skew at short maturity, the convexity of this smile lowers the ratio.
2. **ATM level**: $\sigma_{\mathrm{loc}}(t, 0) \to \sigma_{\mathrm{imp}}(\text{ATM})$ only as $T \to 0$; with a convex smile the local vol at the money
   is below the implied one (the curvature term $\frac12 \partial_{yy} w$ in $g$: the ATM option also sees the higher local vols of
   the wings).
3. **Flat smile**: a flat implied surface gives a flat local vol equal to it.

In [ ]:
def lv_row(T):
    return sigma[np.argmin(np.abs(t_lv - T))]


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, i in zip(axes, (1, 3, 6)):
    T = T_NODES[i]
    iv_slice = np.sqrt(np.maximum(W[i], 1e-12) / T)
    ax.plot(y, iv_slice, label="implied")
    ax.plot(y_lv[sel], lv_row(T)[sel], label="local")
    ax.plot(Y_NODES[i], IV[i], "k.", ms=5)
    ax.set(title=f"T = {T:.2f}", xlabel="y")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.show()

rows = []
for i, T in enumerate(T_NODES[:5]):
    iv_s = np.sqrt(np.maximum(W[i], 1e-12) / T)
    j = np.argmin(np.abs(y)); k = np.argmin(np.abs(y_lv))
    skew_iv = (iv_s[j + 5] - iv_s[j - 5]) / (y[j + 5] - y[j - 5])
    row = lv_row(T)
    skew_lv = (row[k + 2] - row[k - 2]) / (y_lv[k + 2] - y_lv[k - 2])
    rows.append((T, iv_s[j], row[k], skew_iv, skew_lv, skew_lv / skew_iv))
print(pd.DataFrame(rows, columns=["T", "ATM implied", "ATM local", "implied skew", "local skew", "ratio"]).round(3).to_string(index=False))

In [ ]:
# flat smile: the same Gatheral routine on a constant 25% surface
flat = np.full_like(IV, 0.25)
sigma_flat, _, diag_flat = tq.gatheral_local_vol(flat, T_NODES, tf.constant(Y_NODES), y_lv)
print(f"flat 25% surface -> local vol in [{sigma_flat.numpy().min():.6f}, {sigma_flat.numpy().max():.6f}]", diag_flat)

### Implied vol as an average of local vol
Gatheral (3.8)-(3.11): the implied variance is the time average of the local variance over the paths that end at the
strike, weighted by a Brownian-bridge density from the spot to the strike (centre $\tfrac tT\,y_K$ in forward log-moneyness,
variance $\sigma^2 t (T-t)/T$):
$$\sigma_{\mathrm{imp}}^2(K, T) \approx \frac1T\int_0^T \E_{\text{bridge}}\big[\sigma_{\mathrm{loc}}^2(t, y_t)\big]\,\mathrm{d}t
\;\approx\; \frac1T\int_0^T \sigma_{\mathrm{loc}}^2\big(t, \tfrac tT\,y_K\big)\,\mathrm{d}t\ \ \text{(peak only)}.$$
The peak-only version explains the skew rule (the path to a strike only gets half-way in $y$ on average) and why
short-dated wings need far higher local vols; it misses the convexity of the local variance around the path, which the
bridge average restores: within about 1 vol point near the money (0.3-1), cruder in the short-dated downside wing,
where a Gaussian bridge in $y$ is a poor picture of the paths.

In [ ]:
GH_X, GH_W = np.polynomial.hermite_e.hermegauss(40)                # Gauss-Hermite for a standard normal
GH_W = GH_W / GH_W.sum()


def path_average(T, yK, iv, bridge=True, n=200):
    total = 0.0
    for t in (np.arange(n) + 0.5) / n * T:
        row = sigma[np.searchsorted(t_lv, t)]
        sd = iv * np.sqrt(t * (T - t) / T) if bridge else 0.0
        total += np.sum(GH_W * np.interp(t / T * yK + sd * GH_X, y_lv, row) ** 2)
    return np.sqrt(total / n)


for bridge in (False, True):
    rows = {f"T={T_NODES[i]:.2f}": [100 * (path_average(T_NODES[i], yk, iv, bridge) - iv) for yk, iv in zip(Y_NODES[i], IV[i])]
            for i in (1, 3, 5, 6)}
    print(("bridge average" if bridge else "peak only") + " - market implied vol (vol points), by strike / spot:")
    print(pd.DataFrame(rows, index=[f"{k:.0%}" for k in K_NODES / S0]).T.round(2).to_string(), "\n")

## 5. Monte Carlo
$X_t = S_t / F(t)$ with $\mathrm{d}X/X = \sigma_{\mathrm{loc}}(t, \ln X)\,\mathrm{d}W$ (log-Euler), $S_t = F(t)\,X_t$: the forward is the market one
at every date, dividends included. 20k paths, 50 steps a year, observed at the surface maturities.

In [ ]:
N_PATHS, STEPS = 20_000, 50
T_MAX = T_NODES[-2]                                       # up to 3Y
obs = [T for T in T_NODES if T <= T_MAX]
grid = np.unique(np.concatenate([np.arange(1, int(STEPS * T_MAX) + 1) / STEPS, obs]))
dw = tf.random.stateless_normal([N_PATHS, len(grid)], seed=[42, 0], dtype=tf.float64)
paths = model.evolve(grid, dw).numpy()                    # [paths, steps], martingale correction on
raw = tq.LocalVolatilityModel.from_equity(eq, martingale_correction=False).evolve(grid, dw).numpy()

fwd = np.array([float(eq.forward(date_of(t))) for t in grid])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid, paths[:100].T, color="steelblue", alpha=0.15, lw=0.6)
ax.plot(grid, raw.mean(axis=0), color="tomato", lw=2, label="MC mean (no correction)")
ax.plot(grid, fwd, "k--", lw=1.5, label="Equity.forward")
ax.set(xlabel="t", ylabel="S", title="Local vol paths and forward (drops at the ex-dates)",
       ylim=(0.3 * S0, 2.2 * S0))
ax.legend()
plt.show()
cols = [int(np.argmin(np.abs(grid - T))) for T in obs]
err = (raw[:, cols].mean(axis=0) - fwd[cols]) / (raw[:, cols].std(axis=0) / np.sqrt(N_PATHS))
print("forward error without correction, in standard errors:", np.round(err, 2))
print("with correction (exact):", np.abs(paths[:, cols].mean(axis=0) / fwd[cols] - 1).max())

### The smile repriced
OTM options on the simulated paths (puts below the forward, calls above) back to implied vols, against the market nodes:
the local vol model reprices the surface it was built from, up to two errors:
- the **Monte Carlo error**: with 20k paths, a few tenths of a vol point, changing with the seed;
- the **discretisation bias** (log-Euler, local vol bilinear on its grid): with 50 steps a year the repriced smile is
  about 0.2 vol points high on average (at most 0.5); with 200 steps a year it is gone (both checked with 100k paths).

In [ ]:
def black_otm(F, K, s, T, DF):
    d1 = (np.log(F / K) + 0.5 * s * s * T) / (s * np.sqrt(T))
    call = DF * (F * norm.cdf(d1) - K * norm.cdf(d1 - s * np.sqrt(T)))
    return call - DF * (F - K) if K < F else call


def implied(price, F, K, T, DF):
    try:
        return brentq(lambda s: black_otm(F, K, s, T, DF) - price, 1e-4, 3.0)
    except ValueError:
        return np.nan


rows = {}
for i, T in enumerate(obs):
    if T < 0.4:
        continue
    s_T, F, DF = paths[:, cols[i]], F_NODES[i], float(eq.discount_curve.discount(date_of(T)))
    mc = [implied(DF * np.mean(np.maximum(K - s_T, 0) if K < F else np.maximum(s_T - K, 0)), F, K, T, DF) for K in K_NODES]
    rows[f"T={T:.2f}"] = 100 * (np.array(mc) - IV[i])
print("MC implied - market (vol points), by strike / spot:")
print(pd.DataFrame(rows, index=[f"{k:.0%}" for k in K_NODES / S0]).T.round(2).to_string())

### Forward smile
A forward-start call pays $(S_{T_2}/S_{T_1} - k)^+$: its implied vol (on the forward ratio $F(T_2)/F(T_1)$) is the smile the
model predicts for one year starting in one year. Local vol flattens it: the local skew decays with time, so the
forward smile from 1Y to 2Y is much flatter than today's 1Y smile, while stochastic vol keeps it (Heston, SLV).

In [ ]:
i1, i2 = list(T_NODES).index(1.0), int(np.argmin(np.abs(T_NODES - 2.0)))
ratio = paths[:, cols[i2]] / paths[:, cols[i1]]
f_ratio = F_NODES[i2] / F_NODES[i1]
dT = T_NODES[i2] - T_NODES[i1]
ks = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3])
fwd_smile = [implied(np.mean(np.maximum(k * f_ratio - ratio, 0) if k < 1 else np.maximum(ratio - k * f_ratio, 0)),
                     f_ratio, k * f_ratio, dT, 1.0) for k in ks]
spot_smile = [float(np.sqrt(np.interp(np.log(k), y, W[i1]) / T_NODES[i1])) for k in ks]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(ks, 100 * np.array(spot_smile), "o-", label="today's 1Y smile (market)")
ax.plot(ks, 100 * np.array(fwd_smile), "s-", label="1Y forward smile in 1Y (local vol MC)")
ax.set(xlabel="moneyness k (strike / forward)", ylabel="implied vol (%)", title="Forward smile flattening")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Greeks by AAD
`VanillaMCPricer(model=LocalVolatilityModel)`: the model is rebuilt from the market inside the tape, so the pathwise
delta and the vega **per vol node** come out of one backward pass. The local vol reprices every vanilla, so with the
surface held in absolute strikes its delta is the Black–Scholes sticky-strike one with the **same dividend model** (cash
dividends: `dividend_model="discrete"`; the default continuous model holds the implied yield and gives a smaller delta).
Against Black–Scholes:
the total vega is close, but the local vol puts it on the 1Y nodes around the strike with small +/- contributions on
the neighbours and on the earlier slices (a node moves the local vol, through $\partial_T w$ and the spline, on a region).

In [ ]:
expiry = date_of(1.0)
K = round(float(eq.forward(expiry)), 2)


def call():
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Call,
                            strike=K, underlying=TICKER)


lv_call, bs_call = call(), call()
lv = tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=20_000, n_steps=50, seed=7)
lv.price(lv_call, mkt, autodiff=True)
bs = tq.BlackScholesPricer(dividend_model="discrete")
bs.price(bs_call, mkt, autodiff=True)
sources = [eq.spot, vs.volatility_matrix]
(d_lv, v_lv), (d_bs, v_bs) = lv.tape.gradient(lv_call.price, sources), bs.tape.gradient(bs_call.price, sources)
print(f"price  LV {float(lv_call.price):.3f} +/- {float(lv_call.standard_error):.3f}   BS {float(bs_call.price):.3f}")
print(f"delta  LV {float(d_lv):.4f}   BS {float(d_bs):.4f}")
print(f"vega (sum over nodes, per vol point)  LV {float(tf.reduce_sum(v_lv)) / 100:.3f}   BS {float(tf.reduce_sum(v_bs)) / 100:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, v, name in ((axes[0], v_lv, "local vol (MC, AAD)"), (axes[1], v_bs, "Black-Scholes")):
    m = ax.imshow(v.numpy() / 100, aspect="auto", cmap="RdBu_r", vmin=-0.4, vmax=0.4)
    ax.set(title=f"vega per node: {name}", xticks=range(len(K_NODES)), xticklabels=[f"{k:.0%}" for k in K_NODES / S0],
           yticks=range(len(T_NODES)), yticklabels=np.round(T_NODES, 2), xlabel="K / S0")
axes[0].set_ylabel("T")
fig.colorbar(m, ax=axes)
plt.show()

## 6. The limits of local vol
Local vol does one thing perfectly: it reprices **today's** vanilla surface, at every strike and maturity, with one
model. Everything it says beyond that (how the smile moves, what the future smiles look like, the hedge) comes from an
assumption that the market does not support: **the vol is a deterministic function of time and spot**. The limits all
follow from it.

**1. The future smile is too flat (forward smile flattening).** The local skew is steep at short maturities and decays
with time (section 3). A smile seen from a future date $T_1$ is produced by the local vols after $T_1$, which are flat,
so the model predicts flat future smiles (section 5: the 1Y $\to$ 2Y forward smile against today's 1Y). In the market
the skew of a 1Y option is roughly as steep in a year as today. Products that depend on future smiles are mispriced:
forward-start options, cliquets, and the barriers and digitals observed at future dates in autocallables.

**2. The smile moves the wrong way when the spot moves.** In the model the vol at a level $S$ is fixed, so when the spot
falls the stock simply moves into a region of higher local vol. Hagan et al. (*Managing smile risk*, Wilmott, 2002)
show that the implied smile the model predicts after a spot move shifts in the **opposite** direction to the observed
one. So the delta and vega of the model's own dynamics are wrong, and a hedge built on them can be worse than plain
Black–Scholes. In practice the surface is recalibrated every day and the greeks are taken with a market convention
(here sticky strike: the surface held in absolute strikes, section 5), not from the model's dynamics.

**3. The vol has no risk of its own.** The spot and the vol move together, through a deterministic function: there is
no separate vol factor, so no volatility of vol and effectively a perfect spot–vol correlation. Everything that pays
for the randomness of vol (options on variance, volga and vanna effects, the forward skew of point 1) is priced only
through the shape of today's surface. Two models that reprice the same vanillas can give quite different exotic
prices: compare local vol, Heston and stochastic local vol on the same down-and-in put in `down_and_in_put.ipynb`.
That difference is model risk, and local vol gives no handle to measure or price it.

**4. The result depends on the interpolation.** Dupire fixes the local vol only where the call prices are known
everywhere; with 72 quotes the rest is a choice (section 3.5): the spline in strike, linear in time, the wings beyond
the last strikes, a flat implied vol before the first and after the last maturity. Other reasonable choices give
another local vol, the same at the quotes and different in between and in the wings, and the exotics that live there
(far barriers, long maturities) move with it. Side effects of the choices made here: the local vol jumps at every
quoted maturity (constant $\partial_T w$ between maturities), and the closed-form pricers interpolate the same nodes linearly
in vol, so between the nodes the two do not see exactly the same smile.

**5. Noisy quotes and arbitrage.** A second derivative in strike amplifies noise: a small error in one quote moves the
local vol around it a lot, and a quote that creates a butterfly or calendar arbitrage ($g \le 0$ or $\partial_T w \le 0$) makes the
local vol undefined (here floored, with a warning). The same happens day to day: a recalibration with slightly
different quotes can change the local vol surface, and the vega per node (section 5: small positive and negative
contributions around the strike) is noisy. This is why the practice prefers arbitrage-free parametric smiles (SSVI)
or a PDE fit (Andreasen–Huge) before Dupire (section 3, "in practice").

**6. Extreme values at short maturities.** To produce a steep short-dated skew with a diffusion, the local vol has to
be very steep (about twice the implied skew, and more in the wings: the rule of 2). The market's short-dated skew is
largely jump risk; local vol reproduces its prices with implausible vols far from the spot.

**7. Numerics.** The Monte Carlo is a discretisation (log-Euler steps, local vol read on a grid): a bias of about
0.2 vol points with 50 steps a year, on top of the statistical error (section 5).

| limit | consequence | remedy |
|---|---|---|
| flat forward smile | forward-start, cliquets, autocallables mispriced | stochastic vol, stochastic local vol (vol of vol $\times$ mixing) |
| smile moves against the market | poor model hedge | recalibrate, market delta convention, stochastic vol |
| no vol factor | no price for vol risk, exotic prices model-dependent | Heston, SLV: compare models on the product |
| interpolation-dependent | wings and between-node exotics uncertain | arbitrage-free smile (SSVI), PDE fit; test other choices |
| noise and arbitrage | unstable local vol and greeks | clean / smooth the quotes first |
| short-dated skew | extreme local vols | jumps (or stochastic vol with jumps) |

In short: local vol is the right tool when an exotic depends mainly on **today's** smile (a European digital, a barrier
close to expiry) and the standard first model for path-dependent equity products; when it depends on how the smile
**evolves** (forward skew, vol of vol), it has to be checked against a model with stochastic vol, which is what Heston
and the stochastic local vol model of the library are for.

## Exercises
1. Bump one node of the 6M slice by 1 vol point: where does the local vol move (plot the difference of the grids)?
2. Replace 50 by 12 steps a year in the Monte Carlo: how much worse is the 1Y smile in the wings?
3. Why is the local vol at a short maturity in the far wings so much higher than the implied vol there (use the path
   average above)?